# Forge3D no Google Colab ? NVIDIA GPU

1. Escolha **Runtime ? Change runtime type ? GPU NVIDIA** (n?o TPU). 2. Clique **Run all**. 3. Digite seu authtoken ngrok no campo protegido (ou configure a vari?vel `NGROK_AUTHTOKEN` no ambiente). 4. Ao finalizar, receba URLs da interface e do `SKILL.md`, mais uma senha tempor?ria do Forge3D.

**TPU v5e n?o ? compat?vel com este runtime CUDA.** O Colab pode oferecer modelos de GPU e limites variados. Pesos s?o baixados durante o primeiro in?cio; mantenha a sess?o ativa.


## Baixar c?digo automaticamente

O c?digo Forge3D ? clonado do GitHub para `/content/Forge3D`. Opcionalmente use `FORGE3D_SOURCE_ZIP` / `FORGE3D_SOURCE_DIR` para c?digo privado. N?o ? necess?rio anexar dataset.


In [ ]:
import os
import shutil
import stat
import subprocess
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

REPOSITORY_URL = os.environ.get("FORGE3D_REPO_URL", "https://github.com/guell11/Forge3D.git")
GIT_REF = os.environ.get("FORGE3D_GIT_REF", "").strip()
FORGE3D_SOURCE_ZIP = os.environ.get("FORGE3D_SOURCE_ZIP", "").strip()
FORGE3D_SOURCE_DIR = os.environ.get("FORGE3D_SOURCE_DIR", "").strip()
WORK_DIRECTORY = Path("/content")
PROJECT_DIRECTORY = WORK_DIRECTORY / "Forge3D"
MAX_SOURCE_ARCHIVE_BYTES = 512 * 1024 * 1024
MAX_SOURCE_UNPACKED_BYTES = 512 * 1024 * 1024
MAX_SOURCE_MEMBER_BYTES = 128 * 1024 * 1024
MAX_SOURCE_MEMBERS = 20000
REQUIRED_SOURCE_FILES = (
    "backend/app/main.py",
    "backend/requirements.txt",
    "backend/requirements-mcp.txt",
    "backend/requirements-faithc.txt",
    "frontend/package.json",
    "frontend/package-lock.json",
    "instalação/install.sh",
    "scripts/run-t4.sh",
    "scripts/serve_remote.py",
    "scripts/faithc_worker.py",
)


FORBIDDEN_SOURCE_PARTS = {
    ".git", "node_modules", "dist", "runtime", "models", "outputs", "uploads",
    "logs", "artifacts", "__pycache__", ".cache", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".ipynb_checkpoints", "cache", "caches",
}
FORBIDDEN_SOURCE_SUFFIXES = {".zip", ".log", ".glb", ".fbx", ".obj", ".pyc"}


def validate_source_path(raw_name):
    relative_path = PurePosixPath(raw_name.rstrip("/"))
    if (
        not raw_name
        or "\\" in raw_name
        or relative_path.is_absolute()
        or any(part in {"", ".", ".."} for part in relative_path.parts)
        or relative_path.parts[0].endswith(":")
    ):
        raise RuntimeError(f"Caminho inseguro no source: {raw_name!r}")
    if any(
        part.casefold() in FORBIDDEN_SOURCE_PARTS
        or part.casefold().startswith((".env", ".venv"))
        for part in relative_path.parts
    ):
        raise RuntimeError(f"Source contém diretório gerado ou privado: {raw_name}")
    if relative_path.name.casefold() in {".ds_store", "thumbs.db"} or relative_path.suffix.casefold() in FORBIDDEN_SOURCE_SUFFIXES:
        raise RuntimeError(f"Source contém arquivo privado ou gerado: {raw_name}")
    return relative_path


def import_source_zip(source_archive, project_directory, work_directory):
    archive_path = Path(source_archive).expanduser()
    if not archive_path.is_absolute():
        raise RuntimeError("FORGE3D_SOURCE_ZIP precisa ser caminho absoluto do ZIP montado pelo Kaggle.")
    if not archive_path.is_file():
        raise RuntimeError(f"ZIP de source não encontrado: {archive_path}")
    if archive_path.stat().st_size > MAX_SOURCE_ARCHIVE_BYTES:
        raise RuntimeError("ZIP de source excede o limite de 512 MiB.")
    project_directory = Path(project_directory)
    work_directory = Path(work_directory)
    if project_directory.exists() or project_directory.is_symlink():
        raise RuntimeError(
            f"Destino já existe: {project_directory}. Mova ou remova manualmente antes de importar o ZIP; nenhum arquivo foi sobrescrito."
        )
    work_directory.mkdir(parents=True, exist_ok=True)
    staging_directory = Path(tempfile.mkdtemp(prefix=".forge3d-source-", dir=work_directory))
    try:
        seen_paths = set()
        total_unpacked = 0
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            if not members or len(members) > MAX_SOURCE_MEMBERS:
                raise RuntimeError("ZIP vazio ou com quantidade excessiva de arquivos.")
            for member in members:
                raw_name = member.filename
                relative_path = validate_source_path(raw_name)
                if member.file_size > MAX_SOURCE_MEMBER_BYTES:
                    raise RuntimeError(f"Arquivo muito grande no ZIP: {raw_name}")
                total_unpacked += member.file_size
                if total_unpacked > MAX_SOURCE_UNPACKED_BYTES:
                    raise RuntimeError("Source extraído excede o limite de 512 MiB.")
                key = "/".join(relative_path.parts).casefold()
                if key in seen_paths:
                    raise RuntimeError(f"Caminho duplicado no ZIP: {raw_name}")
                seen_paths.add(key)
                file_type = stat.S_IFMT(member.external_attr >> 16)
                if file_type not in {0, stat.S_IFREG, stat.S_IFDIR}:
                    raise RuntimeError(f"ZIP contém link ou arquivo especial: {raw_name}")
                if member.is_dir():
                    (staging_directory / Path(*relative_path.parts)).mkdir(parents=True, exist_ok=True)
                    continue
                target = staging_directory / Path(*relative_path.parts)
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member, "r") as source, target.open("xb") as output:
                    shutil.copyfileobj(source, output, length=1024 * 1024)
        missing = [name for name in REQUIRED_SOURCE_FILES if not (staging_directory / name).is_file()]
        if missing:
            raise RuntimeError("ZIP não contém source Forge3D esperado: " + ", ".join(missing))
        project_directory.parent.mkdir(parents=True, exist_ok=True)
        if project_directory.exists() or project_directory.is_symlink():
            raise RuntimeError(f"Destino apareceu durante a importação: {project_directory}")
        staging_directory.rename(project_directory)
    except BaseException:
        shutil.rmtree(staging_directory, ignore_errors=True)
        raise


def import_source_directory(source_directory, project_directory, work_directory):
    source_path = Path(source_directory).expanduser()
    if not source_path.is_absolute() or source_path.is_symlink() or not source_path.is_dir():
        raise RuntimeError("FORGE3D_SOURCE_DIR precisa ser diretório absoluto montado pelo Kaggle.")
    missing = [name for name in REQUIRED_SOURCE_FILES if not (source_path / name).is_file()]
    if missing:
        raise RuntimeError("Source dir deve conter arquivos esperados na raiz: " + ", ".join(missing))
    project_directory = Path(project_directory)
    work_directory = Path(work_directory)
    if project_directory.exists() or project_directory.is_symlink():
        raise RuntimeError(
            f"Destino já existe: {project_directory}. Mova ou remova manualmente antes de importar; nenhum arquivo foi sobrescrito."
        )
    work_directory.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".forge3d-source-zip-", dir=work_directory) as temporary:
        temporary_archive = Path(temporary) / "mounted-source.zip"
        seen_paths = set()
        total_bytes = 0
        file_count = 0

        def raise_walk_error(error):
            raise error

        with zipfile.ZipFile(temporary_archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
            for current, directories, filenames in os.walk(source_path, topdown=True, followlinks=False, onerror=raise_walk_error):
                current_path = Path(current)
                directories.sort()
                filenames.sort()
                for directory in directories:
                    directory_path = current_path / directory
                    relative_directory = directory_path.relative_to(source_path).as_posix()
                    validate_source_path(relative_directory + "/")
                    metadata = directory_path.lstat()
                    if not stat.S_ISDIR(metadata.st_mode):
                        raise RuntimeError(f"Source contém symlink ou diretório especial: {relative_directory}")
                for filename in filenames:
                    source_file = current_path / filename
                    relative_name = source_file.relative_to(source_path).as_posix()
                    validate_source_path(relative_name)
                    metadata = source_file.lstat()
                    if not stat.S_ISREG(metadata.st_mode):
                        raise RuntimeError(f"Source contém symlink ou arquivo especial: {relative_name}")
                    if metadata.st_size > MAX_SOURCE_MEMBER_BYTES:
                        raise RuntimeError(f"Arquivo muito grande no source: {relative_name}")
                    file_count += 1
                    total_bytes += metadata.st_size
                    if file_count > MAX_SOURCE_MEMBERS or total_bytes > MAX_SOURCE_UNPACKED_BYTES:
                        raise RuntimeError("Source dir excede limite de arquivos ou 512 MiB.")
                    key = relative_name.casefold()
                    if key in seen_paths:
                        raise RuntimeError(f"Caminho duplicado no source: {relative_name}")
                    seen_paths.add(key)
                    descriptor = os.open(source_file, os.O_RDONLY | getattr(os, "O_NOFOLLOW", 0))
                    try:
                        opened = os.fstat(descriptor)
                        if not stat.S_ISREG(opened.st_mode) or opened.st_size != metadata.st_size:
                            raise RuntimeError(f"Arquivo source mudou durante a cópia: {relative_name}")
                        info = zipfile.ZipInfo(relative_name)
                        info.compress_type = zipfile.ZIP_DEFLATED
                        with os.fdopen(descriptor, "rb") as source, archive.open(info, "w") as destination:
                            descriptor = -1
                            shutil.copyfileobj(source, destination, length=1024 * 1024)
                    finally:
                        if descriptor >= 0:
                            os.close(descriptor)
        if temporary_archive.stat().st_size > MAX_SOURCE_ARCHIVE_BYTES:
            raise RuntimeError("Source dir compactado excede limite de 512 MiB.")
        import_source_zip(temporary_archive, project_directory, work_directory)


if FORGE3D_SOURCE_ZIP and FORGE3D_SOURCE_DIR:
    raise RuntimeError("Defina apenas uma origem: FORGE3D_SOURCE_ZIP ou FORGE3D_SOURCE_DIR.")
if FORGE3D_SOURCE_ZIP:
    import_source_zip(FORGE3D_SOURCE_ZIP, PROJECT_DIRECTORY, WORK_DIRECTORY)
    source_revision = "ZIP source local"
elif FORGE3D_SOURCE_DIR:
    import_source_directory(FORGE3D_SOURCE_DIR, PROJECT_DIRECTORY, WORK_DIRECTORY)
    source_revision = "diretório source montado"
elif PROJECT_DIRECTORY.exists():
    if not (PROJECT_DIRECTORY / ".git").exists():
        raise RuntimeError(
            f"Já existe diretório sem checkout Git: {PROJECT_DIRECTORY}. Mova ou remova manualmente; notebook não sobrescreve projetos existentes."
        )
    source_revision = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIRECTORY), "rev-parse", "--short", "HEAD"], text=True
    ).strip()
else:
    command = ["git", "clone", "--depth", "1"]
    if GIT_REF:
        command.extend(["--branch", GIT_REF])
    command.extend([REPOSITORY_URL, str(PROJECT_DIRECTORY)])
    subprocess.run(command, check=True)
    source_revision = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIRECTORY), "rev-parse", "--short", "HEAD"], text=True
    ).strip()

print(f"Projeto pronto em {PROJECT_DIRECTORY}; source: {source_revision}.")


## Conferir GPU e espaço

Notebook falha cedo se GPU física zero não for NVIDIA GPU ou se disco livre não comportar o conjunto GGUF Q4. As dez URLs Q4 definidas pelo instalador responderam HTTP 200 no HEAD usado para preparar este notebook. Somadas, indicam cerca de 6,1 GiB de pesos; deixe espaço extra para runtime, cache e resultados.

In [ ]:
import shutil
import subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpu_rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
).splitlines()
if not gpu_rows:
    raise RuntimeError(f"GPU física 0 precisa ser NVIDIA T4; detectada: {gpu_rows[0] if gpu_rows else 'nenhuma'}")
gpu_memory_mib = int(gpu_rows[0].split(",")[-1].strip().split()[0])
if gpu_memory_mib < 7168:
    raise RuntimeError(f"GPU com {gpu_memory_mib} MiB; o runtime Q4 requer aproximadamente 8 GB VRAM. TPU n?o ? compat?vel.")
free_gib = shutil.disk_usage(WORK_DIRECTORY).free / (1024 ** 3)
if free_gib < 10:
    raise RuntimeError(f"Espaço livre insuficiente em /content: {free_gib:.1f} GiB; libere ao menos 10 GiB.")
print(f"CUDA_VISIBLE_DEVICES=0 · {gpu_rows[0]} · {free_gib:.1f} GiB livres")

## Node.js compatível com Vite

Instalador usa npm para dependências frontend. Se imagem Colab já não tiver Node.js compatível, célula baixa distribuição oficial Node 22 LTS e valida arquivo contra `SHASUMS256.txt` publicado pelo projeto Node antes de adicionar o binário ao PATH.

In [ ]:
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tarfile
import urllib.request
from pathlib import Path


def node_version_tuple(value):
    parts = value.strip().lstrip("v").split(".")
    try:
        return tuple(int(part) for part in parts[:3])
    except ValueError:
        return ()


def node_supports_vite(value):
    version = node_version_tuple(value)
    if len(version) < 2:
        return False
    major, minor = version[:2]
    return (major == 20 and minor >= 19) or (major == 22 and minor >= 12) or major >= 23

current_node = shutil.which("node")
current_version = None
if current_node:
    current_version = subprocess.check_output([current_node, "--version"], text=True).strip()

if not node_supports_vite(current_version or ""):
    if platform.system() != "Linux" or platform.machine().lower() not in {"x86_64", "amd64"}:
        raise RuntimeError("Bootstrap Node 22 suporta Linux x86_64, arquitetura do runtime Kaggle T4.")
    node_cache = WORK_DIRECTORY / ".forge3d-node"
    node_cache.mkdir(parents=True, exist_ok=True)
    manifest_url = "https://nodejs.org/dist/index.json"
    with urllib.request.urlopen(manifest_url, timeout=30) as response:
        releases = json.load(response)
    release = next((item for item in releases if item.get("version", "").startswith("v22.") and item.get("lts")), None)
    if release is None:
        raise RuntimeError("Node.js 22 LTS não apareceu no índice oficial do Node.js.")
    node_release = release["version"]
    archive_name = f"node-{node_release}-linux-x64.tar.xz"
    install_root = node_cache / f"node-{node_release}-linux-x64"
    node_bin = install_root / "bin"
    archive_path = node_cache / archive_name
    if not (node_bin / "node").is_file() or not (node_bin / "npm").exists():
        checksums_url = f"https://nodejs.org/dist/{node_release}/SHASUMS256.txt"
        with urllib.request.urlopen(checksums_url, timeout=30) as response:
            checksum_lines = response.read().decode("ascii").splitlines()
        expected_hash = next((line.split()[0] for line in checksum_lines if line.split()[-1:] == [archive_name]), None)
        if not expected_hash:
            raise RuntimeError("Checksum SHA-256 do arquivo Node não encontrado no manifesto oficial.")
        archive_url = f"https://nodejs.org/dist/{node_release}/{archive_name}"
        with urllib.request.urlopen(archive_url, timeout=180) as response, archive_path.open("wb") as target:
            shutil.copyfileobj(response, target)
        actual_hash = hashlib.sha256(archive_path.read_bytes()).hexdigest()
        if actual_hash != expected_hash:
            archive_path.unlink(missing_ok=True)
            raise RuntimeError("Checksum do Node.js baixado não confere com SHASUMS256.txt.")
        node_cache_resolved = node_cache.resolve()
        with tarfile.open(archive_path, mode="r:xz") as archive:
            members = archive.getmembers()
            for member in members:
                target_path = (node_cache_resolved / member.name).resolve()
                if node_cache_resolved != target_path and node_cache_resolved not in target_path.parents:
                    raise RuntimeError("Arquivo Node contém caminho fora do diretório de instalação.")
                if member.issym() or member.islnk():
                    link_path = (target_path.parent / member.linkname).resolve()
                    if node_cache_resolved != link_path and node_cache_resolved not in link_path.parents:
                        raise RuntimeError("Link Node aponta para fora do diretório de instalação.")
            archive.extractall(node_cache_resolved, members=members)
    os.environ["PATH"] = str(node_bin) + os.pathsep + os.environ.get("PATH", "")
    current_version = subprocess.check_output([str(node_bin / "node"), "--version"], text=True).strip()
    if not node_supports_vite(current_version):
        raise RuntimeError(f"Node.js baixado não satisfaz Vite: {current_version}")

npm_path = shutil.which("npm")
if not npm_path:
    raise RuntimeError("npm não foi encontrado após conferir/bootstrap Node.js.")
print(f"Node.js {current_version}; npm disponível.")

## Instalar runtime e frontend

Instalador Linux existente fixa Q4, 512 px e atlas 1024 para perfil 8gb, baixa o runtime CUDA de trellis.cpp e os pesos do Hugging Face. Notebook mantém saída de instalação compacta e mostra apenas cauda do log quando comando falha.

In [ ]:
from collections import deque
import os
import subprocess


def run_quiet(command, *, cwd, env=None, failure_tail=10000):
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    tail = deque(maxlen=80)
    for line in process.stdout:
        if line.strip():
            tail.append(line.rstrip())
    return_code = process.wait()
    if return_code:
        print("\n".join(tail)[-failure_tail:])
        raise subprocess.CalledProcessError(return_code, command)
    if tail:
        print(tail[-1])

from pathlib import Path
import urllib.request
venv_python = PROJECT_DIRECTORY / ".venv" / "bin" / "python"
clean_env = os.environ.copy()
for key in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
    clean_env.pop(key, None)
clean_env["PYTHONNOUSERSITE"] = "1"
if not venv_python.is_file():
    subprocess.run(["python3", "-I", "-m", "venv", "--without-pip", str(PROJECT_DIRECTORY / ".venv")], check=True, env=clean_env)
probe = subprocess.run([str(venv_python), "-I", "-m", "pip", "--version"], env=clean_env, capture_output=True)
if probe.returncode:
    print("Corrigindo ambiente virtual sem pip...")
    bootstrap = WORK_DIRECTORY / "forge3d-get-pip.py"
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", bootstrap)
    subprocess.run([str(venv_python), "-I", str(bootstrap)], check=True, env=clean_env, stdout=subprocess.DEVNULL)
    subprocess.run([str(venv_python), "-I", "-m", "pip", "--version"], check=True, env=clean_env)

installer = PROJECT_DIRECTORY / "instalação" / "install.sh"
run_quiet(["bash", str(installer), "--profile", "8gb"], cwd=PROJECT_DIRECTORY, env=clean_env)
run_quiet([str(PROJECT_DIRECTORY / "runtime" / "trellis" / "trellis-cli"), "--help"], cwd=PROJECT_DIRECTORY)
print("Instalador e preflight trellis-cli concluíram perfil 8gb/Q4.")

In [ ]:
import re
import os
import subprocess

frontend_directory = PROJECT_DIRECTORY / "frontend"
node_version = subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v")
version_match = re.match(r"(\d+)\.(\d+)", node_version)
if not version_match:
    raise RuntimeError(f"Versão Node.js não reconhecida: {node_version}")
node_major, node_minor = map(int, version_match.groups())
if node_major < 20 or (node_major == 20 and node_minor < 19) or node_major == 21 or (node_major == 22 and node_minor < 12):
    raise RuntimeError(f"Vite 7 requer Node.js 20.19+ ou 22.12+; detectado v{node_version}")

build_environment = os.environ.copy()
build_environment["VITE_API_BASE_URL"] = "/api"
run_quiet(["npm", "run", "build"], cwd=frontend_directory, env=build_environment)
if not (frontend_directory / "dist" / "index.html").is_file():
    raise RuntimeError("Build frontend terminou sem frontend/dist/index.html")
print("Frontend compilado para servir UI, API e arquivos GLB na mesma origem.")

## Opcional: FaithC

FaithC fica desligado por padrão. Célula usa venv separado para evitar alterar Torch/CUDA do backend Forge3D. Para habilitar, mude `ENABLE_FAITHC` para `True` e execute a célula; precisa Internet e downloads próprios. Resolução sugerida para NVIDIA GPU: 256. A instalação completa pode levar alguns minutos e consome disco adicional.

In [ ]:
import json
import os
import subprocess

ENABLE_FAITHC = False
FAITHC_ENVIRONMENT = PROJECT_DIRECTORY / ".venv-faithc"
FAITHC_PYTHON = FAITHC_ENVIRONMENT / "bin" / "python"
FAITHC_WORKER = PROJECT_DIRECTORY / "scripts" / "faithc_worker.py"

if ENABLE_FAITHC:
    requirements = PROJECT_DIRECTORY / "backend" / "requirements-faithc.txt"
    if not requirements.is_file() or not FAITHC_WORKER.is_file():
        raise RuntimeError("FaithC worker ou backend/requirements-faithc.txt não foi encontrado nesta revisão.")
    backend_python = PROJECT_DIRECTORY / ".venv" / "bin" / "python"
    if not FAITHC_PYTHON.is_file():
        subprocess.run([str(backend_python), "-m", "venv", str(FAITHC_ENVIRONMENT)], check=True)
    faithc_environment = os.environ.copy()
    faithc_environment["CUDA_VISIBLE_DEVICES"] = "0"
    faithc_environment["TORCH_CUDA_ARCH_LIST"] = "7.5"
    run_quiet([str(FAITHC_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel", "packaging", "ninja"], cwd=PROJECT_DIRECTORY)
    run_quiet(
        [str(FAITHC_PYTHON), "-m", "pip", "install", "torch==2.5.1+cu124", "--index-url", "https://download.pytorch.org/whl/cu124"],
        cwd=PROJECT_DIRECTORY,
        env=faithc_environment,
    )
    run_quiet(
        [str(FAITHC_PYTHON), "-m", "pip", "install", "--no-build-isolation", "torch-scatter==2.1.2", "-f", "https://data.pyg.org/whl/torch-2.5.1+cu124.html"],
        cwd=PROJECT_DIRECTORY,
        env=faithc_environment,
    )
    run_quiet([str(FAITHC_PYTHON), "-m", "pip", "install", "--no-build-isolation", "-r", str(requirements)], cwd=PROJECT_DIRECTORY, env=faithc_environment)
    probe = subprocess.run([str(FAITHC_PYTHON), str(FAITHC_WORKER), "probe"], cwd=str(PROJECT_DIRECTORY), env=faithc_environment, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    probe_line = next((line for line in probe.stdout.splitlines() if line.startswith("FORGE3D_FAITHC_PROBE:")), None)
    if probe.returncode != 0 or probe_line is None:
        print("\n".join(probe.stdout.splitlines()[-40:]))
        raise RuntimeError("FaithC probe falhou; imports e alocação CUDA precisam passar antes de habilitar o worker.")
    faithc_status = json.loads(probe_line.split(":", 1)[1])
    if faithc_status.get("ok") is not True or not faithc_status.get("cuda_device"):
        raise RuntimeError(f"FaithC probe não confirmou CUDA T4: {faithc_status.get('detail') or faithc_status.get('cuda_device')}")
    print(f"FaithC probe ok · {faithc_status.get('cuda_device')} · Torch {faithc_status.get('torch_version')}")
else:
    print("FaithC desligado; perfil base TRELLIS Q4/512 permanece ativo.")

## Segredos

No Colab, adicione secrets `NGROK_AUTHTOKEN` e `FORGE3D_ACCESS_TOKEN` em **Add-ons → Secrets** e anexe-os ao notebook. Token de acesso Forge3D deve ter pelo menos 32 caracteres ASCII. Se secret não estiver disponível, célula pede valor com `getpass`; nenhum dos valores vai para stdout ou arquivo.

In [ ]:
from getpass import getpass


def read_secret(label, prompt):
    try:
        from google.colab import userdata
        value = userdata.get(label)
    except Exception:
        value = ""
    if not value:
        value = getpass(prompt)
    return value.strip()

NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "").strip() or read_secret("NGROK_AUTHTOKEN", "ngrok authtoken: ")
import secrets
FORGE3D_ACCESS_TOKEN = secrets.token_urlsafe(36)
print("Token Forge3D gerado automaticamente. Guarde o token exibido na ?ltima c?lula para login e MCP.")
if not NGROK_AUTHTOKEN:
    raise ValueError("ngrok authtoken vazio")
if len(FORGE3D_ACCESS_TOKEN) < 32 or not FORGE3D_ACCESS_TOKEN.isascii():
    raise ValueError("FORGE3D_ACCESS_TOKEN precisa ter ao menos 32 caracteres ASCII")
print("Secrets carregados em memória.")

## Iniciar serviço e conferir autenticação

API e build estático rodam em `127.0.0.1:7860`. Servidor só anuncia pronto depois que `/api/health` retorna estado real `ok` com perfil 8gb. Smoke checks confirmam UI sem token redireciona para login e API sem token retorna 401.

In [ ]:
import atexit
import http.client
import json
import os
import re
import signal
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request

def stop_owned_process(process, process_group_id=None, timeout=15):
    if process is None:
        return
    group_id = process_group_id or process.pid
    try:
        os.killpg(group_id, signal.SIGTERM)
    except ProcessLookupError:
        pass
    deadline = time.monotonic() + timeout
    try:
        process.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        try:
            os.killpg(group_id, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=5)
        return
    while time.monotonic() < deadline:
        try:
            os.killpg(group_id, 0)
        except ProcessLookupError:
            return
        time.sleep(0.2)
    try:
        os.killpg(group_id, signal.SIGKILL)
    except ProcessLookupError:
        pass


SERVER_PORT = 7860
SERVER_LOG = PROJECT_DIRECTORY / "logs" / "kaggle-serve.log"
SERVER_LOG.parent.mkdir(parents=True, exist_ok=True)
if not FORGE3D_ACCESS_TOKEN:
    FORGE3D_ACCESS_TOKEN = read_secret("FORGE3D_ACCESS_TOKEN", "Forge3D access token (32+ ASCII chars): ")

previous_tunnel = globals().get("FORGE3D_TUNNEL")
if previous_tunnel is not None and "ngrok" in globals():
    try:
        ngrok.disconnect(previous_tunnel.url())
    except Exception:
        pass
    FORGE3D_TUNNEL = None
previous_process = globals().get("server_process")
if previous_process is not None:
    stop_owned_process(previous_process, globals().get("server_process_group"))
previous_log = globals().get("server_log_handle")
if previous_log is not None and not previous_log.closed:
    previous_log.close()

service_environment = os.environ.copy()
service_environment.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "ENGINE_PROFILE": "8gb",
    "TRELLIS_CLI_GPU_INDEX": "0",
    "TRELLIS_CLI_REQUIRE_GPU": "1",
    "TRELLIS_DEFAULT_RESOLUTION": "512",
    "FORGE3D_PORT": str(SERVER_PORT),
    "FORGE3D_REQUIRE_T4": "0",
    "FORGE3D_ACCESS_TOKEN": FORGE3D_ACCESS_TOKEN,
    "FORGE3D_REQUIRE_T4": "0",
    "FAITHC_ENABLED": "1" if ENABLE_FAITHC else "0",
    "FAITHC_PYTHON": str(FAITHC_PYTHON),
    "FAITHC_WORKER": str(FAITHC_WORKER),
    "FAITHC_TIMEOUT_SECONDS": "900",
    "FAITHC_PROBE_TIMEOUT_SECONDS": "90",
})
server_log_handle = SERVER_LOG.open("w", encoding="utf-8")
server_process = subprocess.Popen(
    ["bash", str(PROJECT_DIRECTORY / "scripts" / "run-t4.sh")],
    cwd=str(PROJECT_DIRECTORY),
    env=service_environment,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
server_process_group = server_process.pid
atexit.register(stop_owned_process, server_process, server_process_group)
service_environment.pop("FORGE3D_ACCESS_TOKEN", None)


def api_request(path, *, bearer=None, cookie=None, accept="application/json"):
    request_path = path if path.startswith("/") else f"/{path}"
    headers = {"Accept": accept}
    if bearer:
        headers["Authorization"] = f"Bearer {bearer}"
    if cookie:
        headers["Cookie"] = cookie
    request = urllib.request.Request(f"http://127.0.0.1:{SERVER_PORT}{request_path}", headers=headers)
    with urllib.request.urlopen(request, timeout=5) as response:
        return response.status, response.headers, response.read()

try:
    health = None
    deadline = time.monotonic() + 120
    while time.monotonic() < deadline:
        if server_process.poll() is not None:
            break
        try:
            _, _, health_body = api_request("/api/health", bearer=FORGE3D_ACCESS_TOKEN)
            health = json.loads(health_body)
            if health.get("status") == "ok":
                break
        except Exception:
            time.sleep(1)
    if not health or health.get("status") != "ok":
        tail = SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-10000:] if SERVER_LOG.exists() else ""
        print(tail)
        raise RuntimeError("Forge3D API não passou no health check dentro de 120 segundos")
    if health.get("active_profile", {}).get("id") != "8gb":
        raise RuntimeError(f"Perfil inesperado na API: {health.get('active_profile')}")

    try:
        api_request("/api/health")
    except urllib.error.HTTPError as error:
        if error.code != 401:
            raise
    else:
        raise RuntimeError("A API respondeu sem bearer token")

    root_request = urllib.request.Request(
        f"http://127.0.0.1:{SERVER_PORT}/", headers={"Accept": "text/html"}
    )
    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    try:
        urllib.request.build_opener(NoRedirect()).open(root_request, timeout=5)
    except urllib.error.HTTPError as error:
        if error.code != 303 or error.headers.get("Location") != "/_auth/login":
            raise RuntimeError(f"Resposta UI sem sessão inesperada: HTTP {error.code}") from None
    else:
        raise RuntimeError("A UI abriu sem sessão")

    host = f"127.0.0.1:{SERVER_PORT}"
    login_connection = http.client.HTTPConnection("127.0.0.1", SERVER_PORT, timeout=5)
    login_body = urllib.parse.urlencode({"token": FORGE3D_ACCESS_TOKEN})
    login_connection.request(
        "POST",
        "/_auth/login",
        body=login_body,
        headers={
            "Host": host,
            "Origin": f"http://{host}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
    )
    login_response = login_connection.getresponse()
    if login_response.status != 303:
        raise RuntimeError(f"Login rejeitado: HTTP {login_response.status}")
    set_cookie = login_response.getheader("Set-Cookie", "")
    login_connection.close()
    session_cookie = set_cookie.split(";", 1)[0]
    if not session_cookie.startswith("forge3d_session="):
        raise RuntimeError("Login não emitiu cookie de sessão")

    ui_status, _, ui_body = api_request("/", cookie=session_cookie, accept="text/html")
    if ui_status != 200:
        raise RuntimeError(f"UI autenticada retornou HTTP {ui_status}")
    script_match = re.search(rb'<script[^>]+src="([^"]+\.js)"', ui_body)
    if not script_match:
        raise RuntimeError("Build UI não anunciou asset JavaScript para smoke check")
    asset_path = script_match.group(1).decode("utf-8")
    asset_status, _, _ = api_request(asset_path, cookie=session_cookie, accept="*/*")
    if asset_status != 200:
        raise RuntimeError(f"Asset UI autenticado retornou HTTP {asset_status}")
    try:
        api_request(asset_path, accept="*/*")
    except urllib.error.HTTPError as error:
        if error.code != 401:
            raise
    else:
        raise RuntimeError("Asset UI respondeu sem sessão")
    api_request("/api/health", cookie=session_cookie)
    try:
        api_request("/outputs/nao-gerado.glb")
    except urllib.error.HTTPError as error:
        if error.code != 401:
            raise
    else:
        raise RuntimeError("Rota GLB respondeu sem sessão")
    try:
        api_request("/api/jobs/id-inexistente/events", cookie=session_cookie)
    except urllib.error.HTTPError as error:
        if error.code != 404:
            raise RuntimeError(f"EventSource retornou HTTP {error.code}; esperado 404 para job inexistente") from error
    else:
        raise RuntimeError("EventSource retornou sucesso para job inexistente")
    print("API health ok · perfil 8gb · login/cookie HttpOnly ok · API, assets, EventSource e GLB protegidos.")
except BaseException:
    stop_owned_process(server_process, server_process_group)
    if not server_log_handle.closed:
        server_log_handle.close()
    FORGE3D_ACCESS_TOKEN = ""
    raise


## Criar túnel

ngrok Python SDK recebe token direto em memória; notebook não grava `ngrok.yml`, não exporta token em comando shell e não imprime token. URL aponta para UI/API na mesma origem. Acesse-a no navegador e entre com seu `FORGE3D_ACCESS_TOKEN`.

Browser `EventSource` e downloads GLB usam cookie de sessão `HttpOnly` na mesma origem. Bridge MCP pode usar a URL HTTPS como base e `Authorization: Bearer <FORGE3D_ACCESS_TOKEN>` em chamadas server-to-server.

In [ ]:
import importlib
import sys
import inspect

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "").strip() or read_secret("NGROK_AUTHTOKEN", "ngrok authtoken: ")
previous_tunnel = globals().get("FORGE3D_TUNNEL")
if previous_tunnel is not None:
    try:
        if inspect.isawaitable(previous_tunnel):
            previous_tunnel = await previous_tunnel
        closing = ngrok.disconnect(previous_tunnel.url())
        if inspect.isawaitable(closing):
            await closing
    except Exception:
        pass
try:
    run_quiet([sys.executable, "-m", "pip", "install", "ngrok"], cwd=PROJECT_DIRECTORY)
    ngrok = importlib.import_module("ngrok")
except BaseException:
    NGROK_AUTHTOKEN = ""
    stop_owned_process(globals().get("server_process"), globals().get("server_process_group"))
    log_handle = globals().get("server_log_handle")
    if log_handle is not None and not log_handle.closed:
        log_handle.close()
    raise
try:
    FORGE3D_TUNNEL = ngrok.forward(
        f"127.0.0.1:{SERVER_PORT}",
        authtoken=NGROK_AUTHTOKEN,
    )
    if inspect.isawaitable(FORGE3D_TUNNEL):
        FORGE3D_TUNNEL = await FORGE3D_TUNNEL
except Exception:
    NGROK_AUTHTOKEN = ""
    stop_owned_process(globals().get("server_process"), globals().get("server_process_group"))
    log_handle = globals().get("server_log_handle")
    if log_handle is not None and not log_handle.closed:
        log_handle.close()
    raise RuntimeError("ngrok não conseguiu criar endpoint; confira secret, Internet e estado da conta.") from None
NGROK_AUTHTOKEN = ""
from IPython.display import display, Markdown
base_url = FORGE3D_TUNNEL.url().rstrip("/")
skill_path = PROJECT_DIRECTORY / "skills" / "forge3d-kaggle" / "SKILL.md"
if not skill_path.is_file():
    raise RuntimeError("SKILL.md do agente n?o encontrado no projeto instalado")
# Link HTTPS acess?vel pelo agente, sem colocar segredo na URL.
# A rota autenticada de arquivos ? disponibilizada pelo gateway.
print("\n=== FORGE3D PRONTO ===")
print("Interface:", base_url)
print("Token de acesso (guarde em local seguro):", FORGE3D_ACCESS_TOKEN)
print("SKILL.md (link para enviar ao agente):", base_url + "/agent/SKILL.md")
display(Markdown(f"**[Abrir Forge3D]({base_url})**  |  **[SKILL.md para o agente]({base_url}/agent/SKILL.md)**"))


## Encerrar

Execute célula para fechar listener ngrok e serviço local ao terminar. Encerrar ou reiniciar sessão Colab também derruba recursos, mas célula de cleanup fecha-os de forma ordenada.

In [ ]:
async def stop_forge3d():
    tunnel = globals().get("FORGE3D_TUNNEL")
    tunnel_sdk = globals().get("ngrok")
    if tunnel is not None and tunnel_sdk is not None:
        try:
            if inspect.isawaitable(tunnel):
                tunnel = await tunnel
            closing = tunnel_sdk.disconnect(tunnel.url())
            if inspect.isawaitable(closing):
                await closing
        except Exception:
            pass
        globals()["FORGE3D_TUNNEL"] = None
    process = globals().get("server_process")
    stop_process = globals().get("stop_owned_process")
    if process is not None and stop_process is not None:
        stop_process(process, globals().get("server_process_group"))
    log_handle = globals().get("server_log_handle")
    if log_handle is not None and not log_handle.closed:
        log_handle.close()
    print("ngrok e Forge3D encerrados.")

stop_forge3d()